---
## Step 6: List All Pipeline Outputs

```python
# List everything uploaded
for key in s3.list(BUCKET_NAME):
    print(f"  s3://{BUCKET_NAME}/{key}")
```

## What I Built
- A complete data pipeline: raw → quality check → clean → features → report
- Simulated S3 storage (swap `LocalS3` with real boto3 when ready)
- Data quality validation before processing
- Parquet output for efficient downstream use

## What I Learned
- 

In [ ]:
# TODO: Generate and upload a quality report as JSON
# Include:
# - Pipeline run timestamp
# - Raw row count, processed row count, rows dropped
# - Data quality issues found
# - List of keys written to S3

quality_report.update({
    'run_timestamp': datetime.now().isoformat(),
    # TODO: add your findings
})

# Upload as JSON
# s3.put(BUCKET_NAME, 'reports/pipeline_run_latest.json', json.dumps(quality_report, indent=2))
# print(json.dumps(quality_report, indent=2))

---
## Step 5: Quality Report

In [ ]:
# TODO: Create analytics features
# - Monthly revenue by product and region
# - Customer tier × product revenue matrix
# - Running 7-day revenue total
# - Create a features DataFrame with all engineered columns

# TODO: Upload to S3
# Key: features/sales_features_2024.parquet

---
## Step 4: Feature Engineering

In [ ]:
# TODO: Clean the data
# - Remove exact duplicates
# - Remove rows with negative quantity
# - Impute missing unit_price with median by product
# - Calculate revenue = quantity * unit_price * (1 - discount_pct/100)
# - Extract date features: year, month, day_of_week, is_weekend
# - Add tier_rank (Bronze=1, Silver=2, Gold=3, Platinum=4)
# clean_df = ...

# TODO: Upload processed data to S3
# Key: processed/sales_2024_clean.parquet (use parquet for processed!)

---
## Step 3: Transform & Clean

In [ ]:
# TODO: Download from S3 and run quality checks
# df = s3.get(BUCKET_NAME, 'raw/sales_2024.csv')

# TODO: Calculate and print:
# - Total rows, duplicate count
# - Missing values per column (count + %)
# - Negative quantities (invalid)
# - Price range (min, max, mean)

# TODO: Create a quality_report dict with all findings
quality_report = {}

---
## Step 2: Data Quality Check

In [ ]:
# TODO: Upload raw_df to S3
# Key: raw/sales_2024.csv
# Use s3.put(BUCKET_NAME, 'raw/sales_2024.csv', raw_df)
# Print confirmation

---
## Step 1: Upload Raw Data to S3

In [ ]:
def generate_raw_sales(n=5000, seed=42):
    """Generate raw sales data with realistic messiness."""
    np.random.seed(seed)
    start = datetime(2024, 1, 1)
    dates = [start + timedelta(days=int(x)) for x in np.random.randint(0, 365, n)]
    df = pd.DataFrame({
        'order_id': [f'ORD-{i:05d}' for i in range(n)],
        'date': dates,
        'product': np.random.choice(['Laptop', 'Phone', 'Tablet', 'Watch', 'Headphones'], n),
        'region': np.random.choice(['North', 'South', 'East', 'West'], n),
        'quantity': np.random.randint(1, 10, n),
        'unit_price': np.random.choice([299, 499, 699, 999, 1299], n) + np.random.normal(0, 20, n),
        'discount_pct': np.random.choice([0, 5, 10, 15, 20], n, p=[0.5, 0.2, 0.15, 0.1, 0.05]),
        'customer_tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], n, p=[0.4, 0.3, 0.2, 0.1]),
    })
    # Inject messiness
    df.loc[np.random.choice(n, 50, replace=False), 'unit_price'] = np.nan
    df.loc[np.random.choice(n, 30, replace=False), 'quantity'] = -1
    df = pd.concat([df, df.sample(20, random_state=99)])  # duplicates
    return df.reset_index(drop=True)

raw_df = generate_raw_sales()
print(f"Raw data shape: {raw_df.shape}")
raw_df.head()

In [ ]:
# Configuration
BUCKET_NAME = f"ml-pipeline-{os.getenv('USER', 'user')}-{datetime.now().strftime('%Y%m')}"
USE_LOCAL_SIMULATOR = True  # Set to False if you have real AWS credentials

class LocalS3:
    """Local S3 simulator for practice without AWS costs."""
    def __init__(self, base='/tmp/s3-sim'):
        self.base = Path(base); self.base.mkdir(parents=True, exist_ok=True)
    def put(self, bucket, key, df_or_str):
        p = self.base / bucket / key; p.parent.mkdir(parents=True, exist_ok=True)
        if isinstance(df_or_str, pd.DataFrame):
            if key.endswith('.parquet'): df_or_str.to_parquet(p, index=False)
            else: df_or_str.to_csv(p, index=False)
        else: p.write_text(str(df_or_str))
        return f"s3://{bucket}/{key}"
    def get(self, bucket, key):
        p = self.base / bucket / key
        if key.endswith('.parquet'): return pd.read_parquet(p)
        elif key.endswith('.csv'): return pd.read_csv(p)
        return p.read_text()
    def list(self, bucket, prefix=''):
        bp = self.base / bucket
        return [str(p.relative_to(bp)) for p in bp.rglob('*') if p.is_file() and str(p.relative_to(bp)).startswith(prefix)]

s3 = LocalS3() if USE_LOCAL_SIMULATOR else None
print(f"Storage backend: {'Local Simulator' if USE_LOCAL_SIMULATOR else 'AWS S3'}")

In [ ]:
import boto3, io, json, os, time
import pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv('../../../.env')
AWS_REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')

# Mini-Project 1: S3 Data Pipeline

## Goal
Build a complete data pipeline that:
1. Generates raw sales data locally
2. Uploads to S3 (real or simulated)
3. Applies transformations
4. Saves processed data back to S3
5. Generates a quality report

## Estimated Time: 2 hours